In [27]:
import os
import asyncio
import re
import jsonlines
from bs4 import BeautifulSoup
import nest_asyncio
import pythonmonkey as pm

from playwright.async_api import async_playwright
from playwright._impl._errors import TimeoutError

from time import perf_counter

mml2tex = pm.require('./mml2latex')
nest_asyncio.apply()
mml2tex

{'mml2latex': <pythonmonkey.JSFunctionProxy object at 0x766d970681f0>}

In [2]:
def get_cleaned_text(html_text:str, as_list:bool=False) -> str:
    soup = BeautifulSoup(html_text, 'html.parser')
    for mml_container in soup.find_all('mjx-container'):
        mml = mml_container.find('mjx-assistive-mml').find('math')
        mml_container.replace_with(mml2tex.mml2latex(str(mml)).strip())

    def clean_artifact(s:str) -> str:
        s = re.sub(r'\xa0', ' ', s)
        s = re.sub(r'</?p>', '', s)
        s = re.sub(r'\s,', ',', s)
        s = re.sub(r'=', ' = ', s)
        s = re.sub(r'\s+', ' ', s)
        s = s.strip()
        return s
    
    if as_list:
        return [clean_artifact(p.get_text()) for p in soup.find_all('p')]
    else:
        return ' '.join([clean_artifact(p.get_text()) for p in soup.find_all('p')])

In [204]:
async def main(url:str):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto(url, wait_until='commit')
        # print(await page.title())
        # print()
        # print(await page.locator('css=.qbq_text_question').text_content())

        problem_locator = await page.locator('div.qbq_text_question > div.html_text').all()
        problems = []
        for problem_located in problem_locator:
            problem = await problem_located.inner_html()
            problems.append(problem)

        # print(await page.locator('css=.qbq_text_solution').inner_html())
        # print(await page.locator('#answer1.html_text').inner_html())

        solution_locator = await page.locator('div.qbq_text_solution > div.html_text').all()
        print(f"Solutions Found: {len(solution_locator)}")
        solutions = []
        for solution_located in solution_locator:
            solution = await solution_located.inner_html()
            solutions.append(solution)
        print(f"Solutions Extracted: {len(solutions)}")

        # print(await page.get_by_text(' Concept: ').text_content())
        # print(await page.locator('script[type="application/ld+json"]').text_content())
        # info = await page.locator('script[type="application/ld+json"]').inner_html()

        concept = await page.get_by_text('Concept:').text_content()
        concept = concept.replace('Concept:', '').strip()

        next_url = await page.locator('a.next').get_attribute('href')
        current_exercise_details = await page.locator('div.qps_name').inner_text()
        await browser.close()
        
        # info = json.loads(info)
        # print(info[1]['mainEntity']['text'])
        # print()
        # print(print(info[1]['mainEntity']['acceptedAnswer']['text']))

        questions = [get_cleaned_text(p) for p in problems]
        solutions = [get_cleaned_text(s, as_list=True) for s in solutions]
        print(f"Solutions Processed: {len(solutions)}")
        print(f"Problem:\n{questions}\n\nSolution:\n{solutions}\n\nConcept:\n{concept}\n\nNext URL:\n{next_url}\n\nExercise Details:\n{current_exercise_details}")

In [205]:
asyncio.run(main("https://www.shaalaa.com/question-bank-solutions/in-an-election-the-successful-candidate-registered-5-77-500-votes-and-his-nearest-rival-secured-3-48-700-votes-by-what-margin-did-the-successful-candidate-win-the-election-larger-number-of-digits-7-and-above_323858"))

Solutions Found: 1
Solutions Extracted: 1
Solutions Processed: 1
Problem:
['In an election, the successful candidate registered 5,77,500 votes and his nearest rival secured 3,48,700 votes. By what margin did the successful candidate win the election?']

Solution:
[['Votes received by successful candidate = 5,77,500', 'Votes received by rival votes = 3,48,700', 'Difference of votes = Votes received by successful candidate – Votes received by rival votes', '= 5,77,500 − 3,48,700', '= 2,28,800', 'Hence, the successful candidate won the election by 2,28,800 votes.']]

Concept:
Larger Number of Digits 7 and Above

Next URL:
/question-bank-solutions/kirti-bookstore-sold-books-worth-2-85-891-in-the-first-week-of-june-and-books-worth-4-00-768-in-the-second-week-of-the-month-how-much-was-the-sale-for-the-two-weeks-together-larger-number-of-digits-7-and-above_323864#ref=chapter&id=255013

Exercise Details:
Chapter 1: Knowing Our Numbers - Exercise 1.2 [Page 16]


In [206]:
asyncio.run(main("https://www.shaalaa.com/question-bank-solutions/if-sina-34-calculate-cos-a-and-tan-a-trigonometric-ratios_7150"))

Solutions Found: 1
Solutions Extracted: 1
Solutions Processed: 1
Problem:
['If `sin A = 3/4`, calculate cos A and tan A.']

Solution:
[['Let ΔABC be a right-angled triangle, right-angled at point B.', '', 'Given that,', 'sin A = 3/4', '`(BC)/(AC) = 3/4`', 'Let BC be 3k. Therefore, AC will be 4k, where k is a positive integer.', 'Applying Pythagoras theorem in ΔABC, we obtain', 'AC2 = AB2 + BC2', '(4k)2 = AB2 + (3k)2', '16k 2 − 9k 2 = AB2', '7k 2 = AB2', 'AB = `sqrt7k`', '`cos A = ("Side adjacent to"angleA)/"Hypotenuse"`', '∴ cos A = `(AB)/(AC) = sqrt(7k)/(4k) = sqrt7/4`', '`tan A = ("Side adjacent to"angleA)/("Side adjacent to"angleA)`', '` = (BC)/(AB) = (3k)/(sqrt7k) = 3/sqrt7`']]

Concept:
Trigonometric Ratios

Next URL:
/question-bank-solutions/given-15-cot-a-8-find-sin-a-and-sec-a-trigonometric-ratios-and-its-reciprocal_7152#ref=chapter&id=51598

Exercise Details:
Chapter 8: Introduction to Trigonometry - Exercise 8.1 [Page 181]


In [207]:
asyncio.run(main("https://www.shaalaa.com/question-bank-solutions/evaluate-the-following-in-the-simplest-form-sin-60-cos-30-cos-60-sin-30-trigonometric-ratios-of-some-special-angles_5867#ref=chapter&id=51611"))


Solutions Found: 2
Solutions Extracted: 2
Solutions Processed: 2
Problem:
['Evaluate the following in the simplest form: sin 60º cos 30º + cos 60º sin 30º', 'Evaluate the following: sin 60° cos 30° + cos 60° sin 30°']

Solution:
[['sin 60º cos 30º + cos 60º sin 30º', '` = \\frac{\\sqrt{3}}{2}\\times \\frac{\\sqrt{3}}{2}+\\frac{1}{2}\\times\\frac{1}{2} = \\frac{3}{4}+\\frac{1}{4} = 1`'], ['sin 60° cos 30° + cos 60° sin 30° .......(i)', 'By trigonometric ratios we have,', '`sin 60^2 = sqrt3/2 sin 30^@ = 1/2`', '`cos 30^@ = sqrt3/2 cos 60^@ = 1/2`', 'Substituting above values in (i), we get', '`sqrt3/2 . sqrt3/2 + 1/2 . 1/2`', '` = 3/4 + 1/4 = 4/4 = 1`']]

Concept:
Trigonometric Ratios of Some Special Angles

Next URL:
/question-bank-solutions/evaluate-the-following-2tan2-45-cos2-30-sin2-60-trigonometric-ratios-of-some-special-angles_7290#ref=chapter&id=51612

Exercise Details:
Chapter 8: Introduction to Trigonometry - Exercise 8.2 [Page 187]


In [138]:
async def next_page_test(url:str):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        print('Loading Page...')
        await page.goto(url, wait_until='commit')
        print('Page Loaded...')
        exercise_details = await page.locator('div.qps_name').inner_text()
        exercise_details = re.sub(r"\[Page \d+\]", "", exercise_details).strip()
        print(f'\nExercise: {exercise_details}')

        for i in range(100):
            current_qn = await page.locator('a.qps_number').inner_text()
            current_exercise_details = await page.locator('div.qps_name').inner_text()
            current_exercise_details = re.sub(r"\[Page \d+\]", "", current_exercise_details).strip()
            if exercise_details != current_exercise_details:
                print(f'\nExercise: {current_exercise_details}')
                exercise_details = current_exercise_details

            # print(f'{current_qn: <6} | {next_hurl}')
            print(f'{current_qn: <6} |')

            try:
                await page.locator('a.next').wait_for(timeout=1000, state='visible')
            except TimeoutError:
                print('No more questions')
                break

            await page.locator('a.next').click()

        await browser.close()
        

In [139]:
asyncio.run(next_page_test("https://www.shaalaa.com/question-bank-solutions/if-sina-34-calculate-cos-a-and-tan-a-trigonometric-ratios_7150"))

Loading Page...
Page Loaded...

Exercise: Chapter 8: Introduction to Trigonometry - Exercise 8.1
Q 3    |
Q 4    |
Q 5    |
Q 6    |
Q 7.1  |
Q 7.2  |
Q 8    |
Q 9    |
Q 10   |
Q 11.1 |
Q 11.2 |
Q 11.3 |
Q 11.4 |
Q 11.5 |

Exercise: Chapter 8: Introduction to Trigonometry - Exercise 8.2
Q 1.1  |
Q 1.2  |
Q 1.3  |
Q 1.4  |
Q 1.5  |
Q 2.1  |
Q 2.2  |
Q 2.3  |
Q 2.4  |
Q 3    |
Q 4.1  |
Q 4.2  |
Q 4.3  |
Q 4.4  |
Q 4.5  |

Exercise: Chapter 8: Introduction to Trigonometry - Exercise 8.3
Q 1.1  |
Q 1.2  |
Q 1.3  |
Q 1.4  |
Q 2.1  |
Q 2.2  |
Q 3    |
Q 4    |
Q 5    |
Q 6    |
Q 7    |

Exercise: Chapter 8: Introduction to Trigonometry - Exercise 8.4
Q 1    |
Q 2    |
Q 3.1  |
Q 3.2  |
Q 4.1  |
Q 4.2  |
Q 4.3  |
Q 4.4  |
Q 5.01 |
Q 5.02 |
Q 5.03 |
Q 5.04 |
Q 5.05 |
Q 5.06 |
Q 5.07 |
Q 5.08 |
Q 5.09 |
Q 5.1  |
No more questions


In [14]:
async def popup_test(url:str):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        print('Loading Page...')
        await page.goto(url, wait_until='commit')
        print('Page Loaded...')

        # await page.route("**/*", lambda route: route.abort() if route.request.url.startswith("https://googleads.")  else route.continue_())

        async def popup_handler():
            print('Popup Handler')
            print('Popup Found')
            await page.get_by_role("button", name='x').click()
            print('Popup Clicked')

        await page.add_locator_handler(
            page.get_by_role("button", name='x'),
            popup_handler
        )

        print(await page.get_by_role("button", name='x').count())
        print('Sleeping for 10 seconds...')
        await asyncio.sleep(10)
        print('Woke up...')

        # if await page.get_by_role("button", name='x').count() > 0:
        #     print('Popup Found')
        #     await page.get_by_role("button", name='x').click()
        #     print('Popup Clicked')

        problem_locator = await page.locator('div.qbq_text_question > div.html_text').all()
        problems = []
        for problem_located in problem_locator:
            await problem_located.click()
            problem = await problem_located.inner_html()
            problems.append(problem)

        print(problems)
        await asyncio.sleep(5)
        await browser.close()

asyncio.run(popup_test("https://www.shaalaa.com/question-bank-solutions/if-sina-34-calculate-cos-a-and-tan-a-trigonometric-ratios_7150"))

Loading Page...
Page Loaded...
0
Sleeping for 10 seconds...
Woke up...
Popup Handler
Popup Found
Popup Clicked
['<p>If&nbsp;<mjx-container class="MathJax" jax="CHTML" style="font-size: 119.6%; position: relative;"><mjx-math class="MJX-TEX" aria-hidden="true"><mjx-mstyle><mjx-mrow><mjx-mi class="mjx-n"><mjx-c class="mjx-c73"></mjx-c><mjx-c class="mjx-c69"></mjx-c><mjx-c class="mjx-c6E"></mjx-c></mjx-mi><mjx-mi class="mjx-i" space="2"><mjx-c class="mjx-c1D434 TEX-I"></mjx-c></mjx-mi></mjx-mrow><mjx-mo class="mjx-n" space="4"><mjx-c class="mjx-c3D"></mjx-c></mjx-mo><mjx-mfrac space="4"><mjx-frac type="d"><mjx-num><mjx-nstrut type="d"></mjx-nstrut><mjx-mn class="mjx-n"><mjx-c class="mjx-c33"></mjx-c></mjx-mn></mjx-num><mjx-dbox><mjx-dtable><mjx-line type="d"></mjx-line><mjx-row><mjx-den><mjx-dstrut type="d"></mjx-dstrut><mjx-mn class="mjx-n"><mjx-c class="mjx-c34"></mjx-c></mjx-mn></mjx-den></mjx-row></mjx-dtable></mjx-dbox></mjx-frac></mjx-mfrac></mjx-mstyle></mjx-math><mjx-assistive-mml 

In [3]:
async def scrape_ncert_chapter(start_url:str, class_no:str):

    # Start Async Browser
    async with async_playwright() as p:

        # Launch Browser (with display)
        browser = await p.chromium.launch(headless=False)

        # Create a new page
        page = await browser.new_page()
        await page.goto(start_url, wait_until='commit')
        await page.route("**/*", lambda route: route.abort() if route.request.url.startswith("https://googleads.")  else route.continue_())

        # Extract the Current Exercise Details
        exercise_details = await page.locator('div.qps_name').inner_text()
        exercise_details = re.sub(r"\[Page \d+\]", "", exercise_details, re.IGNORECASE).strip()

        # Find the Chapter Number, Chapter Name and Exercise Number
        match = re.search(r'Chapter (\d+): (.+?) - Exercise (\d+\.\d+)', exercise_details, re.IGNORECASE)
        if match:
            chapter_number = int(match.group(1))
            chapter_name = match.group(2)
            exercise_number = float(match.group(3))

        # # Initialize the Loop
        # print(f"Scraping Started for:\n\tClass: {class_no}\n\tChapter: {chapter_number} - {chapter_name}\n\nExercise: {exercise_number}")

        # Loop through all the questions
        while True:
            # Extract the Problem
            problem_locator = await page.locator('div.qbq_text_question > div.html_text').all()
            problems = []
            for problem_located in problem_locator:
                problem = await problem_located.inner_html()
                problems.append(problem)

            # Extract the Solution
            solution_locator = await page.locator('div.qbq_text_solution > div.html_text').all()
            solutions = []
            for solution_located in solution_locator:
                solution = await solution_located.inner_html()
                solutions.append(solution)

            # Extract the Concept
            concept = await page.get_by_text('Concept:').text_content()
            concept = concept.replace('Concept:', '').strip()

            # Extract the Current Question Number
            current_que_no = await page.locator('a.qps_number').inner_text()

            # Check current exercise details
            current_exercise_details = await page.locator('div.qps_name').inner_text()
            current_exercise_details = re.sub(r"\[Page \d+\]", "", current_exercise_details, re.IGNORECASE).strip()
            try:
                exercise_number = re.search(r'Exercise (\d+\.\d+)', current_exercise_details, re.IGNORECASE).group(1)
            except AttributeError:
                print(f'Error: {current_exercise_details}')
                exercise_number = "Error"

            if exercise_details != current_exercise_details:
                # print(f'\nExercise: {exercise_number}')
                exercise_details = current_exercise_details


            # Collect all data
            data = {
                'class': class_no,
                'chapter_number': chapter_number,
                'chapter_name': chapter_name,
                'exercise_number': exercise_number,
                'question_number': current_que_no,
                'problem': [get_cleaned_text(p) for p in problems],
                'solution': [get_cleaned_text(s, as_list=True) for s in solutions],
                'concept': concept
            }

            # Save the data
            with jsonlines.open(os.path.join('..', 'data', 'info_collection', f'class_{class_no}.jsonl'), mode='a') as writer:
                writer.write(data)

            # print(f'{current_qn: <6} | {next_hurl}')
            print(f'Scraped: {current_que_no: <10} | Exercise: {exercise_number}')

            try:
                await page.locator('a.next').wait_for(timeout=1000, state='visible')
            except TimeoutError:
                print('No more questions')
                break

            await page.locator('a.next').click()
            await page.route("**/*", lambda route: route.abort() if route.request.url.startswith("https://googleads.")  else route.continue_())

        await browser.close()

In [3]:
class_6_start_urls = [
    "https://www.shaalaa.com/question-bank-solutions/1-lakh-______-ten-thousand-larger-number-of-digits-7-and-above_323796#ref=chapter&id=254947",
    "https://www.shaalaa.com/question-bank-solutions/write-the-next-three-natural-numbers-after-10999-concept-natural-numbers_324078#ref=chapter&id=255230",
    "https://www.shaalaa.com/question-bank-solutions/write-all-the-factors-of-the-following-number-24-factors-and-multiples_324300#ref=chapter&id=255461",
    "https://www.shaalaa.com/question-bank-solutions/use-the-figure-to-name-five-points-concept-of-points_325267#ref=chapter&id=256477",
    "https://www.shaalaa.com/question-bank-solutions/what-is-the-disadvantage-of-comparing-line-segments-by-mere-observation-measuring-line-segments_325715#ref=chapter&id=256942",
    "https://www.shaalaa.com/question-bank-solutions/write-opposites-of-the-following-increase-in-weight-concept-ordering-integers_327318#ref=chapter&id=258566",
    "https://www.shaalaa.com/question-bank-solutions/write-the-fraction-representing-the-shaded-portion-concept-of-fractions_327451#ref=chapter&id=258705",
    "https://www.shaalaa.com/question-bank-solutions/write-the-following-as-numbers-in-the-given-table-hundreds-100-tens-10-ones-1-tenths-110-concept-of-tenths-hundredths-and-thousandths-in-decimal_327835#ref=chapter&id=259096",
    "https://www.shaalaa.com/question-bank-solutions/in-a-mathematics-test-the-following-marks-were-obtained-by-40-students-arrange-these-marks-in-a-table-using-tally-marksfind-how-many-students-obtained-marks-equal-to-organisation-data_328230#ref=chapter&id=259491",
    "https://www.shaalaa.com/question-bank-solutions/find-the-perimeter-of-the-following-figure-concept-of-perimeter_328303#ref=chapter&id=259565",
    "https://www.shaalaa.com/question-bank-solutions/find-the-rule-which-gives-the-number-of-matchsticks-required-to-make-the-following-matchstick-patterns-use-a-variable-to-write-the-rule-a-pattern-of-letter-t-as-the-idea-of-a-variable_328609#ref=chapter&id=259862",
    "https://www.shaalaa.com/question-bank-solutions/there-are-20-girls-and-15-boys-in-a-class-what-is-the-ratio-of-number-of-girls-to-the-number-of-boys-concept-of-ratio_334072#ref=chapter&id=265422",
    "https://www.shaalaa.com/question-bank-solutions/list-any-four-symmetrical-objects-from-your-home-or-school-concept-of-symmetry_334660#ref=chapter&id=266010",
    "https://www.shaalaa.com/question-bank-solutions/draw-a-circle-of-radius-32-cm-introduction-to-practical-geometry_334961#ref=chapter&id=266303"
]

print(f'Total Chapters: {len(class_6_start_urls)}')

Total Chapters: 14


In [5]:
async def main(urls:list[str], class_no:str):

    # # Start Async Browser
    # async with async_playwright() as p:

    #     # Launch Browser (with display)
    #     browser = await p.chromium.launch(headless=False)
    #     browser_context = await browser.new_context()

    #     # Create New Task with Browser Context
    #     tasks = [scrape_ncert_chapter(browser_context, url, class_no) for url in urls]

    #     await asyncio.gather(*tasks)

    tasks = [scrape_ncert_chapter(url, class_no) for url in urls]
    await asyncio.gather(*tasks)

In [ ]:
asyncio.run(main(class_6_start_urls, class_no='6'))

In [37]:
def convert_seconds(seconds, verbose:bool=False):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    hours, minutes = int(hours), int(minutes)

    if verbose:
        if hours > 0:
            return (f"{hours} hours, {minutes} minutes, {seconds:.2f} seconds")
        elif minutes > 0:
            return (f"{minutes} minutes, {seconds:.2f} seconds")
        else:
            return (f"{seconds:.2f} seconds")
    
    else:
        if hours > 0:
            return (f"{hours}h, {minutes}m, {seconds:.2f}s")
        elif minutes > 0:
            return (f"{minutes}m, {seconds:.2f}s")
        else:
            return (f"{seconds:.2f}s")



In [58]:
async def scrape_ncert_chapter_v2(start_url:str, class_no:str):

    # Start Async Browser
    async with async_playwright() as p:

        chapter_start_time = perf_counter()

        # Launch Browser (with display)
        browser = await p.chromium.launch(headless=False)

        # Create a new page
        page = await browser.new_page()
        await page.goto(start_url, wait_until='commit')
        await page.route("**/*", lambda route: route.abort() if route.request.url.startswith("https://googleads.")  else route.continue_())

        async def ad_popup_handler(popup):
            # print('Popup Handler')
            # print('Popup Found')
            await page.get_by_role("button", name='x').click()
            # print('Popup Clicked')

        # Check for any Ad Popups
        await page.add_locator_handler(page.get_by_role("button", name='x'), ad_popup_handler)

        async def dialog_popup_handler(_):
            # Click the Cancel Button
            await page.get_by_role("button", name='Cancel').click()

        # Check for any Dialog form popup with cancel button
        await page.add_locator_handler(page.get_by_role("button", name='Cancel'), dialog_popup_handler)

        # Extract the Current Exercise Details
        exercise_details = await page.locator('div.qps_name').inner_text()
        exercise_details = re.sub(r"\[Page \d+\]", "", exercise_details, re.IGNORECASE).strip()

        # Find the Chapter Number, Chapter Name and Exercise Number
        match = re.search(r'Chapter (\d+): (.+?) - Exercise (\d+\.\d+)', exercise_details, re.IGNORECASE)
        if match:
            chapter_number = int(match.group(1))
            chapter_name = match.group(2)
            exercise_number = float(match.group(3))
        else:
            match = re.search(r'Chapter (\d+): (.+?) - Miscellaneous Exercise', exercise_details, re.IGNORECASE)
            chapter_number = int(match.group(1))
            chapter_name = match.group(2)
            exercise_number = "Error"

        # # Initialize the Loop
        print(f"Scraping Started for:\n\tClass: {class_no}\n\tChapter: {chapter_number} - {chapter_name}\n\nExercise: {exercise_number}")
        # que_no = ''
        # Loop through all the questions
        while True:
            # Extract the Problem
            problem_start_time = perf_counter()
            problem_locator = await page.locator('div.qbq_text_question > div.html_text').all()
            problems = []
            for problem_located in problem_locator:
                problem = await problem_located.inner_html()
                problems.append(problem)

            # Extract the Solution
            solution_locator = await page.locator('div.qbq_text_solution > div.html_text').all()
            solutions = []
            for solution_located in solution_locator:
                solution = await solution_located.inner_html()
                solutions.append(solution)

            # Extract the Concept
            concept = await page.get_by_text('Concept:').text_content()
            concept = concept.replace('Concept:', '').strip()

            # Check current exercise details
            current_exercise_details = await page.locator('div.qps_name').inner_text()
            current_exercise_details = re.sub(r"\[Page \d+\]", "", current_exercise_details, re.IGNORECASE).strip()
            try:
                exercise_number = re.search(r'Exercise (\d+\.\d+)', current_exercise_details, re.IGNORECASE).group(1)
            except AttributeError:
                print(f'Error: {current_exercise_details}')
                exercise_number = "Error"
            
            # Extract the Current Question Number
            current_que_no = await page.locator('a.qps_number').inner_text()
            # if que_no == current_que_no:
            #     print(f"\nCrawler Stuck at {exercise_number} {current_que_no}. Exiting...")
            #     break
            # else:
            #     que_no = current_que_no


            if exercise_details != current_exercise_details:
                print(f'\nExercise: {exercise_number}')
                exercise_details = current_exercise_details


            # Collect all data
            data = {
                'class': class_no,
                'chapter_number': chapter_number,
                'chapter_name': chapter_name,
                'exercise_number': exercise_number,
                'question_number': current_que_no,
                'problem': [get_cleaned_text(p) for p in problems],
                'solution': [get_cleaned_text(s, as_list=True) for s in solutions],
                'concept': concept
            }

            # Save the data
            with jsonlines.open(os.path.join('..', 'data', 'info_collection', f'class_{class_no}.jsonl'), mode='a') as writer:
                writer.write(data)

            # print(f'{current_qn: <6} | {next_hurl}')
            time_str = convert_seconds(perf_counter() - problem_start_time)
            print(f'Scraped: {current_que_no: <10} | Exercise: {exercise_number: <4} | Time: {time_str}')

            try:
                await page.locator('a.next').wait_for(timeout=1000, state='visible')
            except TimeoutError:
                print('No more questions')
                break

            await page.locator('a.next').click()
            await page.route("**/*", lambda route: route.abort() if route.request.url.startswith("https://googleads.")  else route.continue_())

        await browser.close()
        chapter_end_time = perf_counter()


        time_str = convert_seconds(chapter_end_time - chapter_start_time)
        print(f"Time Taken for Chapter {chapter_number}: {time_str}")

In [22]:
class_6_start_urls = [
    # "https://www.shaalaa.com/question-bank-solutions/1-lakh-______-ten-thousand-larger-number-of-digits-7-and-above_323796#ref=chapter&id=254947",
    # "https://www.shaalaa.com/question-bank-solutions/write-the-next-three-natural-numbers-after-10999-concept-natural-numbers_324078#ref=chapter&id=255230",
    # "https://www.shaalaa.com/question-bank-solutions/write-all-the-factors-of-the-following-number-24-factors-and-multiples_324300#ref=chapter&id=255461",
    # "https://www.shaalaa.com/question-bank-solutions/use-the-figure-to-name-five-points-concept-of-points_325267#ref=chapter&id=256477",
    # "https://www.shaalaa.com/question-bank-solutions/what-is-the-disadvantage-of-comparing-line-segments-by-mere-observation-measuring-line-segments_325715#ref=chapter&id=256942",
    # "https://www.shaalaa.com/question-bank-solutions/write-opposites-of-the-following-increase-in-weight-concept-ordering-integers_327318#ref=chapter&id=258566",
    # "https://www.shaalaa.com/question-bank-solutions/write-the-fraction-representing-the-shaded-portion-concept-of-fractions_327451#ref=chapter&id=258705",
    # "https://www.shaalaa.com/question-bank-solutions/write-the-following-as-numbers-in-the-given-table-hundreds-100-tens-10-ones-1-tenths-110-concept-of-tenths-hundredths-and-thousandths-in-decimal_327835#ref=chapter&id=259096", Exercise 8.1 Q 1(a)- Replaced
    "https://www.shaalaa.com/question-bank-solutions/write-the-following-decimals-in-the-place-value-table-194-place-value-in-the-context-of-decimal-fraction_327837#ref=chapter&id=259098", # Exercise 8.1 Q 2(a) - Replaced
    "https://www.shaalaa.com/question-bank-solutions/in-a-mathematics-test-the-following-marks-were-obtained-by-40-students-arrange-these-marks-in-a-table-using-tally-marksfind-how-many-students-obtained-marks-equal-to-organisation-data_328230#ref=chapter&id=259491",
    "https://www.shaalaa.com/question-bank-solutions/find-the-perimeter-of-the-following-figure-concept-of-perimeter_328303#ref=chapter&id=259565",
    "https://www.shaalaa.com/question-bank-solutions/find-the-rule-which-gives-the-number-of-matchsticks-required-to-make-the-following-matchstick-patterns-use-a-variable-to-write-the-rule-a-pattern-of-letter-t-as-the-idea-of-a-variable_328609#ref=chapter&id=259862",
    "https://www.shaalaa.com/question-bank-solutions/there-are-20-girls-and-15-boys-in-a-class-what-is-the-ratio-of-number-of-girls-to-the-number-of-boys-concept-of-ratio_334072#ref=chapter&id=265422",
    "https://www.shaalaa.com/question-bank-solutions/list-any-four-symmetrical-objects-from-your-home-or-school-concept-of-symmetry_334660#ref=chapter&id=266010",
    "https://www.shaalaa.com/question-bank-solutions/draw-a-circle-of-radius-32-cm-introduction-to-practical-geometry_334961#ref=chapter&id=266303"
]

print(f'Total Chapters: {len(class_6_start_urls)}')

Total Chapters: 7


In [23]:
for start_url in class_6_start_urls:
    asyncio.run(scrape_ncert_chapter_v2(start_url, class_no='6'))

Scraping Started for:
	Class: 6
	Chapter: 8 - Decimals

Exercise: 8.1
Scraped: Q 2. (a)   | Exercise: 8.1
Scraped: Q 2. (b)   | Exercise: 8.1
Scraped: Q 2. (c)   | Exercise: 8.1
Scraped: Q 2. (d)   | Exercise: 8.1
Scraped: Q 3. (a)   | Exercise: 8.1
Scraped: Q 3. (b)   | Exercise: 8.1
Scraped: Q 3. (c)   | Exercise: 8.1
Scraped: Q 3. (d)   | Exercise: 8.1
Scraped: Q 3. (e)   | Exercise: 8.1
Scraped: Q 4. (a)   | Exercise: 8.1
Scraped: Q 4. (b)   | Exercise: 8.1
Scraped: Q 4. (c)   | Exercise: 8.1
Scraped: Q 4. (d)   | Exercise: 8.1
Scraped: Q 4. (e)   | Exercise: 8.1
Scraped: Q 4. (f)   | Exercise: 8.1
Scraped: Q 4. (g)   | Exercise: 8.1
Scraped: Q 4. (h)   | Exercise: 8.1
Scraped: Q 4. (i)   | Exercise: 8.1
Scraped: Q 4. (j)   | Exercise: 8.1
Scraped: Q 4. (k)   | Exercise: 8.1
Scraped: Q 5. (a)   | Exercise: 8.1
Scraped: Q 5. (b)   | Exercise: 8.1
Scraped: Q 5. (c)   | Exercise: 8.1
Scraped: Q 5. (d)   | Exercise: 8.1
Scraped: Q 5. (e)   | Exercise: 8.1
Scraped: Q 5. (f)   | Exercise

In [44]:
class_7_start_urls = [
    "https://www.shaalaa.com/question-bank-solutions/following-number-line-shows-temperature-degree-celsius-c-different-places-particular-day-observe-this-number-line-write-temperature-places-marked-it-concept-integers_16032#ref=chapter&id=9705",
    "https://www.shaalaa.com/question-bank-solutions/solve-fractions-2-3by5-concept-of-fractions_16118#ref=chapter&id=9786",
    "https://www.shaalaa.com/question-bank-solutions/find-range-heights-any-ten-students-your-class-arithmetic-mean-range_16292#ref=chapter&id=9963",
    "https://www.shaalaa.com/question-bank-solutions/complete-last-column-table-x-3-0-value-x-3-concept-of-equation_17177#ref=chapter&id=45931",
    "https://www.shaalaa.com/question-bank-solutions/find-complement-following-angles-1-related-angles-complementary-angles_16150#ref=chapter&id=9818",
    "https://www.shaalaa.com/question-bank-solutions/in-pqr-d-is-the-mid-point-of-qr-pm-is-_________-pd-is-__________-is-qm-mr-altitudes-of-a-triangle_17053#ref=chapter&id=10153",
    "https://www.shaalaa.com/question-bank-solutions/complete-the-following-statement-two-line-segments-are-congruent-if-__________-congruence-among-line-segments_17153#ref=chapter&id=10253",
    "https://www.shaalaa.com/question-bank-solutions/find-ratio-rs-5-50-paise-concept-equivalent-ratios_17100#ref=chapter&id=10200",
    "https://www.shaalaa.com/question-bank-solutions/list-five-rational-numbers-between-1-0-rational-numbers-between-two-rational-numbers_16929#ref=chapter&id=10029",
    "https://www.shaalaa.com/question-bank-solutions/draw-line-say-ab-take-point-c-outside-it-through-c-draw-line-parallel-ab-using-ruler-compasses-only-construction-line-parallel-given-line-through-point-not-line_16341#ref=chapter&id=10012",
    "https://www.shaalaa.com/question-bank-solutions/the-length-breadth-rectangular-piece-land-are-500-m-300-m-respectively-find-its-area-area-of-square_16996#ref=chapter&id=10096",
    "https://www.shaalaa.com/question-bank-solutions/get-algebraic-expressions-following-cases-using-variables-constants-arithmetic-operation-subtraction-z-y-algebraic-expressions_17296#ref=chapter&id=10395",
    "https://www.shaalaa.com/question-bank-solutions/find-value-of-2exponent6-concept-of-exponents_16201#ref=chapter&id=9871",
    "https://www.shaalaa.com/question-bank-solutions/copy-figure-punched-holes-find-axes-symmetry-following-1-lines-symmetry-regular-polygons_17395#ref=chapter&id=10494",
    "https://www.shaalaa.com/question-bank-solutions/identify-nets-which-can-be-used-make-cubes-cut-out-copies-nets-try-it-1-plane-figures-and-solid-shapes_17585#ref=chapter&id=10684"
]

print(f'Total Chapters: {len(class_7_start_urls)}')

class_no = '7'
start_time = perf_counter()
for start_url in class_7_start_urls:
    asyncio.run(scrape_ncert_chapter_v2(start_url, class_no=class_no))
end_time = perf_counter()

time_str = convert_seconds(end_time - start_time, verbose=True)
print('\n\n')
print(f"Total Time Taken for Class {class_no}: {time_str}")

Total Chapters: 15
Scraping Started for:
	Class: 7
	Chapter: 1 - Integers

Exercise: 1.1
Scraped: Q 1.1      | Exercise: 1.1  | Time: 0.08s
Scraped: Q 1.2      | Exercise: 1.1  | Time: 0.18s
Scraped: Q 1.3      | Exercise: 1.1  | Time: 0.21s
Scraped: Q 1.4      | Exercise: 1.1  | Time: 0.25s
Scraped: Q 2        | Exercise: 1.1  | Time: 0.29s
Scraped: Q 3        | Exercise: 1.1  | Time: 0.27s
Scraped: Q 4        | Exercise: 1.1  | Time: 0.33s
Scraped: Q 5        | Exercise: 1.1  | Time: 0.26s
Scraped: Q 6        | Exercise: 1.1  | Time: 0.26s
Scraped: Q 7        | Exercise: 1.1  | Time: 0.36s
Scraped: Q 8.1      | Exercise: 1.1  | Time: 0.26s
Scraped: Q 8.2      | Exercise: 1.1  | Time: 0.27s
Scraped: Q 8.3      | Exercise: 1.1  | Time: 0.25s
Scraped: Q 8.4      | Exercise: 1.1  | Time: 0.12s
Scraped: Q 9.1      | Exercise: 1.1  | Time: 0.21s
Scraped: Q 9.2      | Exercise: 1.1  | Time: 0.22s
Scraped: Q 9.3      | Exercise: 1.1  | Time: 0.46s
Scraped: Q 9.4      | Exercise: 1.1  | Time:

In [46]:
class_8_start_urls = [
    "https://www.shaalaa.com/question-bank-solutions/using-appropriate-properties-find-23-35-52-32-16-properties-rational-numbers-commutative-property-of-rational-numbers_14950#ref=chapter&id=8616",
    "https://www.shaalaa.com/question-bank-solutions/solve-x-2-7-solving-equations-which-have-linear-expressions-on-one-side-and-numbers-on-the-other-side_14991#ref=chapter&id=8658",
    "https://www.shaalaa.com/question-bank-solutions/given-here-are-some-figures-classify-each-of-them-on-the-basis-of-the-following-a-simple-curve-b-simple-closed-curve-c-polygon-d-convex-polygon-econcave-polygon-concept-of-quadrilaterals-sides-adjacent-sides-opposite-sides-angle-adjacent-angles-and-opposite-angles_15058#ref=chapter&id=8725"
    "https://www.shaalaa.com/question-bank-solutions/for-which-these-would-you-use-histogram-show-data-give-reasons-each-graphical-representation-of-data-as-histograms_15136#ref=chapter&id=8803",
    "https://www.shaalaa.com/question-bank-solutions/what-will-be-the-unit-digit-of-the-square-of-the-given-number-81-properties-of-square-numbers_15156#ref=chapter&id=8823",
    "https://www.shaalaa.com/question-bank-solutions/the-following-number-are-not-perfect-cubes-216-some-interesting-patterns-of-cube-numbers_15265#ref=chapter&id=8932",
    "https://www.shaalaa.com/question-bank-solutions/find-the-ratio-of-the-speed-of-a-cycle-15-km-per-hour-to-the-speed-of-scooter-30-km-per-hour-concept-of-ratio_15299#ref=chapter&id=8966",
    "https://www.shaalaa.com/question-bank-solutions/identify-terms-their-coefficients-following-expressions-5xyz2-3zy-terms-factors-and-coefficients-of-expression_15335#ref=chapter&id=9002",
    "https://www.shaalaa.com/question-bank-solutions/a-square-rectangular-field-measurements-given-figure-have-same-perimeter-which-field-has-larger-area-mensuration_15445#ref=chapter&id=9112",
    "https://www.shaalaa.com/question-bank-solutions/evaluate-3-2-powers-with-negative-exponents_15480#ref=chapter&id=9147",
    "https://www.shaalaa.com/question-bank-solutions/following-are-the-car-parking-charges-near-a-railway-station-up-to-4-hours-rs-60-8-hours-rs-100-12-hours-rs-140-24-hours-rs-180-check-if-the-parking-char-concept-direct-proportion_15544#ref=chapter&id=9208",
    "https://www.shaalaa.com/question-bank-solutions/find-the-common-factors-of-the-terms-12x-36-factorisation-by-taking-out-common-factors_15573#ref=chapter&id=9235",
]

print(f'Total Chapters: {len(class_8_start_urls)}')
class_no = '8'
start_time = perf_counter()
for start_url in class_8_start_urls:
    asyncio.run(scrape_ncert_chapter_v2(start_url, class_no=class_no))
end_time = perf_counter()

time_str = convert_seconds(end_time - start_time, verbose=True)
print('\n\n')
print(f"Total Time Taken for Class {class_no}: {time_str}")

Total Chapters: 12
Scraping Started for:
	Class: 8
	Chapter: 1 - Rational Numbers

Exercise: 1.1
Scraped: Q 1.1      | Exercise: 1.1  | Time: 0.07s
Scraped: Q 1.2      | Exercise: 1.1  | Time: 0.57s
Scraped: Q 2.1      | Exercise: 1.1  | Time: 0.29s
Scraped: Q 2.2      | Exercise: 1.1  | Time: 0.56s
Scraped: Q 2.3      | Exercise: 1.1  | Time: 0.34s
Scraped: Q 2.4      | Exercise: 1.1  | Time: 0.39s
Scraped: Q 2.5      | Exercise: 1.1  | Time: 0.33s
Scraped: Q 3.1      | Exercise: 1.1  | Time: 0.53s
Scraped: Q 3.2      | Exercise: 1.1  | Time: 0.37s
Scraped: Q 4.1      | Exercise: 1.1  | Time: 0.55s
Scraped: Q 4.2      | Exercise: 1.1  | Time: 0.39s
Scraped: Q 4.3      | Exercise: 1.1  | Time: 0.30s
Scraped: Q 4.4      | Exercise: 1.1  | Time: 0.63s
Scraped: Q 4.5      | Exercise: 1.1  | Time: 0.43s
Scraped: Q 4.6      | Exercise: 1.1  | Time: 0.21s
Scraped: Q 5.1      | Exercise: 1.1  | Time: 0.54s
Scraped: Q 5.2      | Exercise: 1.1  | Time: 0.18s
Scraped: Q 5.3      | Exercise: 1.1 

Error: Page.goto: Protocol error (Page.navigate): Cannot navigate to invalid URL
Call log:
navigating to "", waiting until "commit"


In [47]:
class_9_start_urls = [
    "https://www.shaalaa.com/question-bank-solutions/is-zero-rational-number-can-you-write-it-form-p-q-where-p-q-are-integersand-q-0-introduction-real-number_6346#ref=chapter&id=3566",
    "https://www.shaalaa.com/question-bank-solutions/which-of-the-following-expression-is-polynomial-in-one-variable-and-which-is-not-state-the-reason-for-your-answer-4x2-3x-7-polynomials-one-variable_6395#ref=chapter&id=3419",
    "https://www.shaalaa.com/question-bank-solutions/how-will-you-describe-position-table-lamp-your-study-table-another-person-coordinate-geometry_6529#ref=chapter&id=2696",
    "https://www.shaalaa.com/question-bank-solutions/the-cost-notebook-twice-cost-pen-write-linear-equation-two-variables-represent-this-statement-linear-equation-in-one-variable_6541#ref=chapter&id=3295",
    "https://www.shaalaa.com/question-bank-solutions/only-one-line-can-pass-through-a-single-point-euclid-s-definitions-axioms-postulates_6590#ref=chapter&id=2620",
    "https://www.shaalaa.com/question-bank-solutions/in-given-figure-lines-ab-cd-intersect-o-if-aoc-boe-70-and-bod-40-find-boe-reflex-coe-pairs-angles_6603#ref=chapter&id=1392",
    "https://www.shaalaa.com/question-bank-solutions/in-quadrilateral-acbd-ac-ad-ab-bisects-a-see-given-figure-show-that-abc-abd-what-can-you-say-about-bc-bd-criteria-for-congruence-of-triangles_6621#ref=chapter&id=3119",
    "https://www.shaalaa.com/question-bank-solutions/the-angles-quadrilateral-are-ratio-3-5-9-13-find-all-angles-quadrilateral-another-condition-quadrilateral-be-parallelogram_6708#ref=chapter&id=3403",
    "https://www.shaalaa.com/question-bank-solutions/which-following-figures-lie-same-base-between-same-parallels-such-case-write-common-base-two-parallels-corollary-a-rectangle-and-a-parallelogram-on-the-same-base-and-between-the-same-parallels-are-equal-in-area_6753#ref=chapter&id=1491",
    "https://www.shaalaa.com/question-bank-solutions/the-centre-circle-lies-in-interior-of-circle-concept-of-circle-centre-radius-diameter-arc-sector-chord-segment-semicircle-circumference-interior-and-exterior-concentric-circles_6880#ref=chapter&id=2078",
    "https://www.shaalaa.com/question-bank-solutions/construct-angle-90-initial-point-given-ray-justify-construction-basic-constructions_6931#ref=chapter&id=1380",
    "https://www.shaalaa.com/question-bank-solutions/a-traffic-signal-board-indicating-school-ahead-equilateral-triangle-side-a-find-area-signal-board-using-heron-s-formula-if-its-perimeter-180-cm-what-will-be-area-signal-board-area-of-a-triangle-by-herons-formula_7142#ref=chapter&id=2739",
    "https://www.shaalaa.com/question-bank-solutions/a-plastic-box-15-m-long-125-m-wide-65-cm-deep-be-made-it-be-open-top-ignoring-thickness-plastic-sheet-determine-surface-area-of-a-cuboid_6945#ref=chapter&id=3404",
    "https://www.shaalaa.com/question-bank-solutions/give-five-examples-data-that-you-can-collect-day-day-life-collecting-data_7180#ref=chapter&id=2520",
    "https://www.shaalaa.com/question-bank-solutions/in-cricket-match-batswoman-hits-boundary-6-times-out-30-balls-she-plays-find-probability-that-she-did-not-hit-boundary-probability-experimental-approach_7114#ref=chapter&id=1905"
]

print(f'Total Chapters: {len(class_9_start_urls)}')
class_no = '9'
start_time = perf_counter()
for start_url in class_9_start_urls:
    asyncio.run(scrape_ncert_chapter_v2(start_url, class_no=class_no))
end_time = perf_counter()

time_str = convert_seconds(end_time - start_time, verbose=True)
print('\n\n')
print(f"Total Time Taken for Class {class_no}: {time_str}")

Total Chapters: 15
Scraping Started for:
	Class: 9
	Chapter: 1 - Number Systems

Exercise: 1.1
Scraped: Q 1        | Exercise: 1.1  | Time: 0.07s
Scraped: Q 2        | Exercise: 1.1  | Time: 0.54s
Scraped: Q 3        | Exercise: 1.1  | Time: 0.39s
Scraped: Q 4.1      | Exercise: 1.1  | Time: 0.25s
Scraped: Q 4.2      | Exercise: 1.1  | Time: 0.57s
Scraped: Q 4.3      | Exercise: 1.1  | Time: 0.45s

Exercise: 1.2
Scraped: Q 1.1      | Exercise: 1.2  | Time: 0.42s
Scraped: Q 1.2      | Exercise: 1.2  | Time: 0.54s
Scraped: Q 1.3      | Exercise: 1.2  | Time: 0.45s
Scraped: Q 2        | Exercise: 1.2  | Time: 0.63s
Scraped: Q 3        | Exercise: 1.2  | Time: 0.35s

Exercise: 1.3
Scraped: Q 1. (i)   | Exercise: 1.3  | Time: 0.38s
Scraped: Q 1. (ii)  | Exercise: 1.3  | Time: 0.38s
Scraped: Q 1. (iii) | Exercise: 1.3  | Time: 0.34s
Scraped: Q 1. (iv)  | Exercise: 1.3  | Time: 0.35s
Scraped: Q 1. (v)   | Exercise: 1.3  | Time: 0.57s
Scraped: Q 1. (vi)  | Exercise: 1.3  | Time: 0.33s
Scraped:

In [48]:
class_10_start_urls = [
    "https://www.shaalaa.com/question-bank-solutions/using-euclid-s-division-algorithm-find-hcf-135-225-euclid-s-division-lemma_5556#ref=chapter&id=51195",
    "https://www.shaalaa.com/question-bank-solutions/the-graphs-y-p-x-are-given-following-figure-some-polynomials-p-x-find-number-zeroes-p-x-each-case-geometrical-meaning-zeroes-polynomial_6384#ref=chapter&id=51158",
    "https://www.shaalaa.com/question-bank-solutions/a-father-tells-his-daughter-seven-years-ago-i-was-seven-times-old-you-were-then-also-three-years-now-i-shall-be-three-times-old-you-will-be-represent-this-situation-algebraically-graphically-graphical-method-of-solution-of-a-pair-of-linear-equations_5598#ref=chapter&id=51247",
    "https://www.shaalaa.com/question-bank-solutions/check-whether-the-following-are-the-quadratic-equations-x-1-2-2-x-3-quadratic-equations_6681#ref=chapter&id=51326",
    "https://www.shaalaa.com/question-bank-solutions/in-which-of-the-following-situation-does-the-list-of-number-involved-make-as-arithmetic-progression-and-why-the-taxi-fare-after-each-km-when-the-fare-is-rs-15-arithmetic-progression_6754#ref=chapter&id=51375",
    "https://www.shaalaa.com/question-bank-solutions/all-circles-are-______-similar-figures_6872#ref=chapter&id=51469",
    "https://www.shaalaa.com/question-bank-solutions/find-the-distance-between-the-following-pairs-of-points-2-3-4-1-distance-formula_7066#ref=chapter&id=51555",
    "https://www.shaalaa.com/question-bank-solutions/in-abc-right-angled-at-b-ab-24-cm-bc-7-m-determine-sin-a-cos-a-trigonometric-ratios_7143#ref=chapter&id=51594",
    "https://www.shaalaa.com/question-bank-solutions/a-circus-artist-is-climbing-a-20-m-long-rope-which-is-tightly-stretched-and-tied-from-the-top-of-a-vertical-pole-to-the-ground-find-the-height-of-the-pole-if-the-angle-heights-and-distances_7349#ref=chapter&id=51656",
    "https://www.shaalaa.com/question-bank-solutions/how-many-tangents-can-a-circle-have-tangent-to-a-circle_7160#ref=chapter&id=51672",
    "https://www.shaalaa.com/question-bank-solutions/the-radii-two-circles-are-19-cm-9-cm-respectively-find-radius-circle-which-has-circumference-equal-sum-circumferences-two-circles-circumference-of-a-circle_7460#ref=chapter&id=51709",
    "https://www.shaalaa.com/question-bank-solutions/2-cubes-each-of-volume-64-cm3-are-joined-end-to-end-find-the-surface-area-of-the-resulting-cuboid-surface-area-combination-solids_7551#ref=chapter&id=51744",
    "https://www.shaalaa.com/question-bank-solutions/a-survey-was-conducted-group-students-part-their-environment-awareness-programme-which-they-collected-following-data-regarding-number-plants-mean-of-grouped-data_7671#ref=chapter&id=51782",
    "https://www.shaalaa.com/question-bank-solutions/probability-of-an-event-e-probability-of-the-event-not-e-_______-probability-theoretical-approach_7203#ref=chapter&id=51807"
]

print(f'Total Chapters: {len(class_10_start_urls)}')
class_no = '10'
start_time = perf_counter()
for start_url in class_10_start_urls:
    asyncio.run(scrape_ncert_chapter_v2(start_url, class_no=class_no))
end_time = perf_counter()

time_str = convert_seconds(end_time - start_time, verbose=True)
print('\n\n')
print(f"Total Time Taken for Class {class_no}: {time_str}")

Total Chapters: 14
Scraping Started for:
	Class: 10
	Chapter: 1 - Real Numbers

Exercise: 1.1
Scraped: Q 1.1      | Exercise: 1.1  | Time: 0.08s
Scraped: Q 1.2      | Exercise: 1.1  | Time: 0.23s
Scraped: Q 1.3      | Exercise: 1.1  | Time: 0.22s
Scraped: Q 2        | Exercise: 1.1  | Time: 0.20s
Scraped: Q 3        | Exercise: 1.1  | Time: 0.19s
Scraped: Q 4        | Exercise: 1.1  | Time: 0.19s
Scraped: Q 5        | Exercise: 1.1  | Time: 0.20s

Exercise: 1.2
Scraped: Q 1.1      | Exercise: 1.2  | Time: 0.25s
Scraped: Q 1.2      | Exercise: 1.2  | Time: 0.26s
Scraped: Q 1.3      | Exercise: 1.2  | Time: 0.28s
Scraped: Q 1.4      | Exercise: 1.2  | Time: 0.75s
Scraped: Q 1.5      | Exercise: 1.2  | Time: 0.72s
Scraped: Q 2.1      | Exercise: 1.2  | Time: 0.75s
Scraped: Q 2.2      | Exercise: 1.2  | Time: 0.37s
Scraped: Q 2.3      | Exercise: 1.2  | Time: 0.37s
Scraped: Q 3.1      | Exercise: 1.2  | Time: 0.46s
Scraped: Q 3.2      | Exercise: 1.2  | Time: 0.43s
Scraped: Q 3.3      | Ex

In [59]:
class_11_start_urls = [
    ## "https://www.shaalaa.com/question-bank-solutions/which-of-the-following-are-sets-justify-our-answer-the-collection-of-all-months-of-a-year-beginning-with-the-letter-j-sets-and-their-representations_13122#ref=chapter&id=7219", Stuck at Ex 1.1 Q 3.4
    # "https://www.shaalaa.com/question-bank-solutions/write-the-following-set-in-roster-form-e-the-set-of-all-letters-in-the-word-trigonometry-sets-and-their-representations_13136#ref=chapter&id=7233" # Ex 1.1 Q3.5
    # "https://www.shaalaa.com/question-bank-solutions/if-x3-1-y-23-53-13-find-the-values-of-x-and-y-sets-ordered-pairs_62979#ref=chapter&id=47131",
    ## "https://www.shaalaa.com/question-bank-solutions/find-the-radian-measure-corresponding-to-the-following-degree-measure-25-concept-angle_13325#ref=chapter&id=7422", #Stuck at Ex 3.3 Q17
    # 'https://www.shaalaa.com/question-bank-solutions/prove-the-following-sinx-sinycosx-cosy-tan-x-y2-trigonometric-functions-sum-difference-two-angles_13368#ref=chapter&id=7466', # Ex 3.3 Q 18
    # "https://www.shaalaa.com/question-bank-solutions/prove-following-using-principle-mathematical-induction-all-n-n-1-3-3-2-3-n-1-3-n-1-2-principle-of-mathematical-induction_13395#ref=chapter&id=7493",
    # "https://www.shaalaa.com/question-bank-solutions/express-the-given-complex-number-in-the-form-a-ib-5i-35i-concept-of-complex-numbers_13419#ref=chapter&id=170831",
    # "https://www.shaalaa.com/question-bank-solutions/solve-24x-100-when-x-is-a-natural-number-algebraic-solutions-linear-inequalities-one-variable-their-graphical-representation_13477#ref=chapter&id=7572",
    ## "https://www.shaalaa.com/question-bank-solutions/how-many-3-digit-numbers-can-be-formed-from-the-digits-1-2-3-4-and-5-assuming-that-repetition-of-the-digits-is-allowed-fundamental-principles-of-counting_13547#ref=chapter&id=7642", Stuck at Ex 7.2 Q5
    # "https://www.shaalaa.com/question-bank-solutions/evaluate-n-n-r-when-n-9-r-5-permutations_13560#ref=chapter&id=7657", # Ex 7.2 Q5.2
    # "https://www.shaalaa.com/question-bank-solutions/expand-the-expression-1-2x-5-binomial-theorem-for-positive-integral-indices_13597#ref=chapter&id=7696",
    # # "https://www.shaalaa.com/question-bank-solutions/write-the-first-five-terms-of-the-sequences-whose-nth-term-is-an-n-n-2-concept-of-sequences_13634#ref=chapter&id=7734", # Stuck at Ex 7.4 Q4
    # "https://www.shaalaa.com/question-bank-solutions/find-the-number-of-ways-of-selecting-9-balls-from-6-red-balls-5-white-balls-and-5-blue-balls-if-each-selection-consists-of-3-balls-of-each-colour-combination_13581#ref=chapter&id=7679" # Ex 7.4 Q5 Stuck Mis Q1
    "https://www.shaalaa.com/question-bank-solutions/how-many-words-with-or-without-meaning-each-of-2-vowels-and-3-consonants-can-be-formed-from-the-letters-of-the-word-daughter-combination_13586#ref=chapter&id=7685",
    "https://www.shaalaa.com/question-bank-solutions/draw-quadrilateral-cartesian-plane-whose-vertices-are-4-5-0-7-5-5-4-2-also-find-its-area-slope-of-a-line_13743#ref=chapter&id=7843",
    "https://www.shaalaa.com/question-bank-solutions/find-equation-circle-centre-0-2-radius-2-concept-circle_13821#ref=chapter&id=7921",
    "https://www.shaalaa.com/question-bank-solutions/a-point-x-axis-what-are-its-y-coordinates-z-coordinates-three-dimensional-geometry-coordinates-of-a-point-in-space_13891#ref=chapter&id=7991",
    "https://www.shaalaa.com/question-bank-solutions/evaluate-given-limit-lim_-x-3-x-3-intuitive-idea-derivatives_13918#ref=chapter&id=8018",
    "https://www.shaalaa.com/question-bank-solutions/which-following-sentences-are-statements-give-reasons-your-answer-all-real-numbers-are-complex-numbers-mathematically-acceptable-statements_14019#ref=chapter&id=8119",
    "https://www.shaalaa.com/question-bank-solutions/find-mean-deviation-about-mean-data-4-7-8-9-10-12-13-17-mean-deviation_14078#ref=chapter&id=8178",
    "https://www.shaalaa.com/question-bank-solutions/describe-sample-space-indicated-experiment-coin-tossed-three-times-random-experiments_14093#ref=chapter&id=8193"
]

print(f'Total Chapters: {len(class_11_start_urls)}')
class_no = '11'
start_time = perf_counter()
for start_url in class_11_start_urls:
    asyncio.run(scrape_ncert_chapter_v2(start_url, class_no=class_no))
end_time = perf_counter()

time_str = convert_seconds(end_time - start_time, verbose=True)
print('\n\n')
print(f"Total Time Taken for Class {class_no}: {time_str}")

Total Chapters: 8
Scraping Started for:
	Class: 11
	Chapter: 7 - Permutations and Combinations

Exercise: Error
Error: Chapter 7: Permutations and Combinations - Miscellaneous Exercise
Scraped: Q 1        | Exercise: Error | Time: 0.08s
Error: Chapter 7: Permutations and Combinations - Miscellaneous Exercise
Scraped: Q 2        | Exercise: Error | Time: 0.23s
Error: Chapter 7: Permutations and Combinations - Miscellaneous Exercise
Scraped: Q 3        | Exercise: Error | Time: 0.54s
Error: Chapter 7: Permutations and Combinations - Miscellaneous Exercise
Scraped: Q 4        | Exercise: Error | Time: 0.48s
Error: Chapter 7: Permutations and Combinations - Miscellaneous Exercise
Scraped: Q 5        | Exercise: Error | Time: 0.27s
Error: Chapter 7: Permutations and Combinations - Miscellaneous Exercise
Scraped: Q 6        | Exercise: Error | Time: 0.63s
Error: Chapter 7: Permutations and Combinations - Miscellaneous Exercise
Scraped: Q 7        | Exercise: Error | Time: 0.56s
Error: Chapte

In [61]:
class_12_start_urls = [
    # "https://www.shaalaa.com/question-bank-solutions/determine-whether-following-relations-are-reflective-symmetric-transitive-relation-r-set-1-2-313-14-defined-r-x-y-3x-y-0-types-of-relations_5877#ref=chapter&id=4294",
    # "https://www.shaalaa.com/question-bank-solutions/find-principal-values-sin-1-1-2-inverse-trigonometric-functions_11810#ref=chapter&id=5136",
    # "https://www.shaalaa.com/question-bank-solutions/in-the-matrix-a-2519-735-2521231-517-the-order-of-the-matrix-order-of-a-matrix_11862#ref=chapter&id=5304",
    # "https://www.shaalaa.com/question-bank-solutions/evaluate-determinants-exercises-1-2-2-4-5-1-introduction-of-determinant_11973#ref=chapter&id=5660",
    # "https://www.shaalaa.com/question-bank-solutions/prove-that-function-f-x-5x-3-continuous-x-0-x-3-x-5-concept-of-continuity_12057#ref=chapter&id=5900",
    # "https://www.shaalaa.com/question-bank-solutions/find-rate-change-area-circle-respect-its-radius-r-when-r-3-cm-rate-of-change-of-bodies-or-quantities_12646#ref=chapter&id=6729",
    "https://www.shaalaa.com/question-bank-solutions/find-anti-derivative-or-integral-following-functions-method-inspection-sin-2x-integration-as-an-inverse-process-of-differentiation_12811#ref=chapter&id=6907",
    "https://www.shaalaa.com/question-bank-solutions/find-area-region-bounded-curve-y2-x-lines-x-1-x-4-x-axis-area-under-simple-curves_13082#ref=chapter&id=7179",
    "https://www.shaalaa.com/question-bank-solutions/determine-order-degree-if-defined-differential-equation-d-4y-dx-4-sin-y-m-0-order-and-degree-of-a-differential-equation_12528#ref=chapter&id=6595",
    "https://www.shaalaa.com/question-bank-solutions/represent-graphically-displacement-40-km-30-east-north-basic-concepts-of-vector-algebra_12453#ref=chapter&id=6520",
    "https://www.shaalaa.com/question-bank-solutions/if-a-line-makes-angles-90-135-45-with-the-x-y-and-z-axes-respectively-find-its-direction-cosines-direction-cosines-and-direction-ratios-of-a-line_103067#ref=chapter&id=90692",
    "https://www.shaalaa.com/question-bank-solutions/solve-following-linear-programming-problems-graphically-maximise-z-3x-4y-subject-constraints-x-y-4-x-0-y-0-linear-programming-problem-and-its-mathematical-formulation_12077#ref=chapter&id=5974",
    "https://www.shaalaa.com/question-bank-solutions/given-that-e-f-are-events-such-that-p-e-06-p-f-03-p-e-f-02-find-p-e-f-p-f-e-conditional-probability_12109#ref=chapter&id=6100"
]

print(f'Total Chapters: {len(class_12_start_urls)}')
class_no = '12'
start_time = perf_counter()
for start_url in class_12_start_urls:
    asyncio.run(scrape_ncert_chapter_v2(start_url, class_no=class_no))
end_time = perf_counter()

time_str = convert_seconds(end_time - start_time, verbose=True)
print('\n\n')
print(f"Total Time Taken for Class {class_no}: {time_str}")

Total Chapters: 7
Scraping Started for:
	Class: 12
	Chapter: 7 - Integrals

Exercise: 7.1
Scraped: Q 1        | Exercise: 7.1  | Time: 0.07s
Scraped: Q 2        | Exercise: 7.1  | Time: 0.22s
Scraped: Q 3        | Exercise: 7.1  | Time: 0.67s
Scraped: Q 4        | Exercise: 7.1  | Time: 0.53s
Scraped: Q 5        | Exercise: 7.1  | Time: 0.27s
Scraped: Q 6        | Exercise: 7.1  | Time: 0.29s
Scraped: Q 7        | Exercise: 7.1  | Time: 0.74s
Scraped: Q 8        | Exercise: 7.1  | Time: 0.23s
Scraped: Q 9        | Exercise: 7.1  | Time: 0.65s
Scraped: Q 10       | Exercise: 7.1  | Time: 0.70s
Scraped: Q 11       | Exercise: 7.1  | Time: 0.73s
Scraped: Q 12       | Exercise: 7.1  | Time: 0.80s
Scraped: Q 13       | Exercise: 7.1  | Time: 1.01s
Scraped: Q 14       | Exercise: 7.1  | Time: 0.50s
Scraped: Q 15       | Exercise: 7.1  | Time: 0.24s
Scraped: Q 16       | Exercise: 7.1  | Time: 0.64s
Scraped: Q 17       | Exercise: 7.1  | Time: 0.59s
Scraped: Q 18       | Exercise: 7.1  | Time